In [23]:
# The City College of New York, City University of New York
# Written by Hasan Suca Kayman and Prof. M. Umit Uyar
# September, 2025
# Data for this example was taken from:
# https://www.kaggle.com/hmavrodiev/london-bike-sharing-dataset

In [24]:
import numpy as np
import pandas as pd
import time
from datetime import datetime
from tensorflow import keras

In [25]:
def print_weights(weights):
    # weights = model.get_weights();
    print('\n******* WEIGHTS OF ANN *******\n')
    for i in range(int(len(weights)/2)):
        print('Weights W%d:\n' %(i), weights[i*2])
        print('Bias b%d:\n' %(i), weights[(i*2)+1])
# END print_weights()

In [26]:
# ANN TRAINING
print('********************************************************************')
print('****  WELCOME TO BIKE SHARING USING ARTIFICIAL NEURAL NETWORKS  ****')
print('********************************************************************')
## OPTION 1: TRAIN A NEW ANN MODEL
train_data_file = 'london_bike_sharing_data.csv'

print('\n********* NOW READING SAMLE DATA USING ', train_data_file,'*********')
time.sleep(3)

## load the training data
df = pd.read_csv(train_data_file)
## the training data contains 6 columns:
##      timestamp - the date and time the sample was recorded
##      new_bikes_shared - number of new bikes shared over the last hour
##      is_weekend - boolean that is 1 (true) if the day is a weekend
##      temp_c - the temperature in Celcius
##      wind_speed - wind speed in km/h
##      weather_code - category of weather: 1 = clear
##                                          2 = scattered clouds
##                                          3 = broken clouds
##                                          4 = cloudy
##                                          7 = rain
##                                         10 = thunderstorm
##                                         26 = snow
##                                         94 = freezing fog
## the timestamp column of df are stored as strings. We want to
## convert each timestamp string into a datetime objects using the
## function datetime.strptime(). The first input of datetime.strptime()
## is the string you want to convert, and the second input is the
## format of the string, where
## %m = month, %d = day, %Y = year, %H = hour, %M = minute.
df

********************************************************************
****  WELCOME TO BIKE SHARING USING ARTIFICIAL NEURAL NETWORKS  ****
********************************************************************

********* NOW READING SAMLE DATA USING  london_bike_sharing_data.csv *********


,timestamp,new_bikes_shared,is_weekend,temp_c,wind_speed,weather_code
0,1/4/2015 0:00,182,1,3.0,6.0,3
1,1/4/2015 1:00,138,1,3.0,5.0,1
2,1/4/2015 2:00,134,1,2.5,0.0,1
3,1/4/2015 3:00,72,1,2.0,0.0,1
4,1/4/2015 4:00,47,1,2.0,6.5,1
...,...,...,...,...,...,...
666,1/31/2015 19:00,565,1,5.0,26.0,4
667,1/31/2015 20:00,447,1,5.0,24.0,4
668,1/31/2015 21:00,305,1,4.5,27.0,3
669,1/31/2015 22:00,276,1,4.0,24.0,4


In [27]:
## create lambda function to perform conversion and return the hour.
get_hour = lambda timestamp: datetime.strptime(timestamp,
                                                '%m/%d/%Y %H:%M').hour
## apply the lambda function to every timestamp in column df['timestamp']
df['time_hour'] = df['timestamp'].apply(get_hour)
df

,timestamp,new_bikes_shared,is_weekend,temp_c,wind_speed,weather_code,time_hour
0,1/4/2015 0:00,182,1,3.0,6.0,3,0
1,1/4/2015 1:00,138,1,3.0,5.0,1,1
2,1/4/2015 2:00,134,1,2.5,0.0,1,2
3,1/4/2015 3:00,72,1,2.0,0.0,1,3
4,1/4/2015 4:00,47,1,2.0,6.5,1,4
...,...,...,...,...,...,...,...
666,1/31/2015 19:00,565,1,5.0,26.0,4,19
667,1/31/2015 20:00,447,1,5.0,24.0,4,20
668,1/31/2015 21:00,305,1,4.5,27.0,3,21
669,1/31/2015 22:00,276,1,4.0,24.0,4,22


In [28]:
## define input matrix X (get rid of columns called timestamp and
## new_bikes_shared)
X = np.array(df.drop(['timestamp','new_bikes_shared'], axis=1))
## define expected output matrix Y
Y = np.array(df['new_bikes_shared'])

In [29]:
# print the first 10 rows of X
X[:10]

array([[1. , 3. , 6. , 3. , 0. ],
       [1. , 3. , 5. , 1. , 1. ],
       [1. , 2.5, 0. , 1. , 2. ],
       [1. , 2. , 0. , 1. , 3. ],
       [1. , 2. , 6.5, 1. , 4. ],
       [1. , 2. , 4. , 1. , 5. ],
       [1. , 1. , 7. , 4. , 6. ],
       [1. , 1. , 7. , 4. , 7. ],
       [1. , 1.5, 8. , 4. , 8. ],
       [1. , 2. , 9. , 3. , 9. ]])

In [30]:
# print the first 10 rows of Y
Y[:10]

array([182, 138, 134,  72,  47,  46,  51,  75, 131, 301])

In [31]:
## create a model for the ANN
model = keras.Sequential()
## add a hidden layer that accepts 5 input features (time_hour, temp_c
## wind_speed, weather_code, is_weekend)
## the hidden layer has 5 neurons.
## Dense means every neuron in the layer connects to every neuron in the
## previous layer.
## define input layer:
model.add(keras.layers.Input(shape=(5,)))
## add the first hidden layer with 10 neurons to the ANN:
model.add(keras.layers.Dense(10, activation='sigmoid'))
## add the second hidden layer with 7 neurons to the ANN:
model.add(keras.layers.Dense(7, activation='sigmoid'))
## add the third hidden layer with 5 neurons to the ANN:
model.add(keras.layers.Dense(5, activation='sigmoid'))
## add an output layer with a single output (new_bikes_shared):
model.add(keras.layers.Dense(1, activation='linear'))

## set the optimization algorithm used for minimizing loss function
## use gradient descent (adam) to minimize error (loss)
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 10)             │            60 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │            77 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 183 (732.00 B)

 Trainable params: 183 (732.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
# print the initial values of weights:
weights = model.get_weights()
print_weights(weights)


******* WEIGHTS OF ANN *******

Weights W0:
 [[-0.04009485 -0.0373494  -0.32679522  0.00141245  0.4911297   0.44098002
  -0.21875101  0.3915314  -0.08215928 -0.3881581 ]
 [ 0.06426632 -0.438164   -0.26713052 -0.32301915 -0.08202279 -0.5101358
   0.35345834  0.2510525   0.06098878  0.23079544]
 [-0.32934868  0.11101085  0.52450925  0.24813849  0.2607348  -0.06621695
   0.2536602   0.44894904 -0.6050217   0.14203918]
 [ 0.51278764  0.40160793  0.13341749  0.2823515  -0.19736445  0.36831194
  -0.40274727 -0.15566114  0.35786426  0.12300837]
 [ 0.5010956   0.35938752  0.45960516 -0.22062007 -0.33340943 -0.27623487
  -0.25051054 -0.1021052  -0.3882934  -0.04399395]]
Bias b0:
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Weights W1:
 [[-0.45045984  0.21074677 -0.14884198  0.40701938  0.08040041 -0.57501453
   0.23997259]
 [ 0.21067083 -0.40719366 -0.02175617 -0.42240644  0.01122326  0.21522778
  -0.34172305]
 [-0.19264552  0.45183146 -0.32222918 -0.20882964 -0.22596845 -0.56219393
   0.13318205]
 [-0.46

In [33]:
## train the ANN model using 200 iterations
model.fit(X, Y, epochs=100, verbose=0)

print('\n\n********** ANN training complete **********\n\n')



********** ANN training complete **********




In [34]:
# print the weighst after training:
weights = model.get_weights();
print_weights(weights)


******* WEIGHTS OF ANN *******

Weights W0:
 [[ 0.23227891 -0.01175948 -0.19974972 -0.16219915  0.7302796   0.68746203
  -0.00281088  0.42462906  0.40158573 -0.25538352]
 [ 0.24052177 -0.38182458 -0.13772473 -0.45901954  0.14753021 -0.203161
   0.53365415  0.2783678   0.5263821   0.26835424]
 [-0.12709449  0.15189049  0.63281846  0.08199413  0.48346546  0.21904594
   0.45936552  0.47483075 -0.03074045  0.21057315]
 [ 0.69288605  0.44565618  0.2606917   0.09358688  0.07640195  0.6438033
  -0.11156761 -0.08346975  0.8003484   0.24574716]
 [ 0.6475375   0.3790006   0.5915469  -0.3581726  -0.04044001  0.03480692
   0.00245457 -0.03924339  0.14086838  0.08096689]]
Bias b0:
 [ 0.21060286  0.05492959  0.17156218 -0.17045014  0.29915625  0.30178952
  0.27558672  0.06857649  0.55917925  0.157016  ]
Weights W1:
 [[-0.11549187  0.64345133  0.23506366  0.7701148   0.05908057 -0.06268626
   0.55935013]
 [ 0.53281397  0.03825105  0.38147703 -0.08024889  0.03226403  0.75626856
  -0.04064709]
 [ 0.12

In [35]:
# PREDICT NUMBER BIKES USING THE TRAINED ANN:
# input('\n\n********** Press ENTER to start using the ANN **********\n\n')
## prompt user for inputs
temp_c = float(input('\n\nEnter temperature in Celcius: \n'))
hour = float(input('Enter hour of the day (military): (0-23) \n'))
is_weekend = input('Is it the weekend? (y/n): \n')
if is_weekend == 'y':
    is_weekend = 1
else:
    is_weekend = 0
wind_speed = float(input('Enter wind speed: (km/h) \n'))
weather_code = int(input('Enter weather code: (1 = clear, 2 = few clouds, '
                + '3 = broken clouds, 4 = cloudy, 7 = rain, '
                + '10 = thunderstorm, 26 = snow, 94 = freezing fog) \n'))

user_input=np.array([[is_weekend, temp_c, wind_speed, weather_code,hour]])
prediction = model.predict(user_input)

## restrict prediction to non-negative values
if prediction < 0:
    prediction = 0.1
else:
    pass

## display prediction
print('\n*****************************************')
print(f'ANN Predicted number of shared bikes: {prediction[0][0]:.0f}', )
print('*****************************************')




Enter temperature in Celcius: 
9
Enter hour of the day (military): (0-23) 
17
Is it the weekend? (y/n): 
n
Enter wind speed: (km/h) 
22
Enter weather code: (1 = clear, 2 = few clouds, 3 = broken clouds, 4 = cloudy, 7 = rain, 10 = thunderstorm, 26 = snow, 94 = freezing fog) 
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step

*****************************************
ANN Predicted number of shared bikes: 11
*****************************************


In [36]:
## ask user if they would like to save the ANN model
choice = ''
while choice not in ['y','n']:
    choice = input('\n\nWould you like to save the ANN model? (y/n): \n')
    if choice == 'y':
        save_name = input('\n\nEnter a name for the save file: \n')
        ## if file name does not end with '.h5', add '.h5' to the file name
        if save_name[-3:] != '.h5':
            save_name += '.h5'
        model.save(save_name)
        print('\n\n')
        print('***** ANN MODEL SUCCESSFULLY SAVED AS '+save_name+' *****')
    elif choice == 'n':
        pass
    else:
        print("Invalid input: Input must be 'y' or 'n'")




Would you like to save the ANN model? (y/n): 
y


Enter a name for the save file: 
8049





***** ANN MODEL SUCCESSFULLY SAVED AS 8049.h5 *****
